# FXS / 03d — Trailing stop on EURUSD-01, EURUSD-02, USDJPY-01, USDJPY-02 (2021 → 2026, compounding)

Iván's rule (2026-11-10), no take-profit:
- **Initial stop −12 pips** from the fill.
- When the trade reaches **+8 pips**, the stop jumps to **+3 pips**.
- It then follows **5 pips behind every new whole pip** of profit, never moving back.

Three versions of every trade, same entries (each structure's frozen start time, every trading day, news days included):

| Version | Exit |
|---|---|
| `BASE` | scheduled end time only (as 03b) |
| `TRAIL+TIME` | trailing stop, or the scheduled end time if no stop is hit first |
| `TRAIL ONLY` | trailing stop only; the trade can run for hours or days. A new entry is skipped while that structure's previous trade is still open |

**Fills (minute data):** entry at the quote at the start time; in each minute the stop is checked against the minute's adverse extreme *before* the favourable extreme can move it (a stop is never raised and hit in the same minute); a minute that opens beyond the stop fills at its open (weekend and news gaps). Costs as 03b: Dukascopy spread × `SPREAD_FACTOR` and `COMMISSION_PER_LOT` per lot round trip. Account: $6,000, each trade sized at entry as L × realised equity (compounding); with `TRAIL ONLY` several trades can be open at once.

**Read with care.** Descriptive: the stop settings were chosen after seeing these years, the structures were picked with 2021–2025 results, and 2026 has been seen (FXS 05/06). EURUSD-01 failed its confirmation.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

E0 = 6_000.0
LEVERAGES = [1, 2, 3, 5, 10]
COMMISSION_PER_LOT = 3.0          # USD per lot, round trip
SPREAD_FACTOR = 0.9               # x Dukascopy spread
IDS = ["EURUSD-01", "EURUSD-02", "USDJPY-01", "USDJPY-02"]
FROZEN = {"EURUSD-01": ("UTC", 75, 195, 1), "EURUSD-02": ("NY", 660, 900, -1),
          "USDJPY-01": ("UTC", 55, 115, -1), "USDJPY-02": ("UTC", 25, 55, 1)}
WINDOWS = ((dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"), (dt.date(2026, 1, 1), dt.date(2026, 11, 6), "holdout"))

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs03d_")
    os.environ.pop("DUKASCOPY_DIR", None)
    LEVERAGES = [1, 5]
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
RULE = F.TrailRule(initial=-12.0, activate=8.0, distance=5.0)
VARIANTS = {"BASE": (F.NO_STOP, True), "TRAIL+TIME": (RULE, True), "TRAIL ONLY": (RULE, False)}
BRANCH = Path.cwd().parent
OUT = Path.cwd() / ("03d_synthetic" if SYNTHETIC else "03d"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220, "display.max_columns", 40)

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(("EURUSD", "USDJPY")):
        F.write_synthetic_candles(base, p, dt.date(2020, 12, 28), dt.date(2026, 11, 6), seed=11 + i, drift_bps=-3.0 if p == "EURUSD" else 0.0,
                                  drift_ny=(4, 8), price=150.0 if p.endswith("JPY") else 1.10)
    cand = pd.DataFrame([{"id": k, "pair": k[:6], "clock": c, "start": s, "end": e, "direction": d} for k, (c, s, e, d) in FROZEN.items()])
else:
    cand = F.load_frozen_candidates(BRANCH).query("id in @IDS").reset_index(drop=True)
for _, r in cand.iterrows():
    assert (r["clock"], int(r["start"]), int(r["end"]), int(r["direction"])) == FROZEN[r["id"]], f"{r['id']} is not the frozen rule"
cand = cand.set_index("id").loc[IDS].reset_index()
display(cand[["id", "pair", "clock", "start", "end", "direction"]].assign(
    rule=[f"{'buy' if d > 0 else 'sell'} {F.slot_label(c, s)} -> {F.slot_label(c, e)}" for c, s, e, d in zip(cand["clock"], cand["start"], cand["end"], cand["direction"])]))

## 1. Minute data (bid/ask open-high-low-close), 2020-12-31 → 2026-11-06

In [ ]:
PAIRS = sorted(set(cand["pair"]))
minutes, days = {}, {}
for p in PAIRS:
    parts = [F.load_minutes_ohlc(p, a, b, st, cfg.min_file_share) for a, b, st in WINDOWS]
    m = pd.concat(parts)
    minutes[p] = F.scale_spread(m[~m.index.duplicated(keep="first")].sort_index(), SPREAD_FACTOR)
    for c in set(cand.loc[cand["pair"] == p, "clock"]):
        days[(p, c)] = np.concatenate([F.build_panel(F.load_minutes(p, a, b, st, min_file_share=cfg.min_file_share), p, c, a, b, cfg.min_coverage).dates
                                       for a, b, st in WINDOWS])
    print(p, len(minutes[p]), "minutes", minutes[p].index[0], "->", minutes[p].index[-1])

## 2. Every trade, three ways

In [ ]:
trades = []
for _, r in cand.iterrows():
    for v, (rule, time_exit) in VARIANTS.items():
        t = F.run_structure(minutes[r["pair"]], r["pair"], r["clock"], int(r["start"]), int(r["end"]), int(r["direction"]),
                            days[(r["pair"], r["clock"])], rule, time_exit, COMMISSION_PER_LOT)
        trades.append(t.assign(structure=r["id"], variant=v))
trades = pd.concat(trades, ignore_index=True)
trades["year"] = pd.to_datetime(trades["date"]).dt.year
trades.to_parquet(OUT / "03d_trades.parquet")
print(trades.groupby(["variant", "structure"]).size().unstack())

## 3. Per structure: pips, win rate, how trades end, how long they last

In [ ]:
def per_structure(t):
    reasons = t["reason"].value_counts(normalize=True)
    return pd.Series({"trades": len(t), "net pips/trade": t["net_pips"].mean(), "win rate": (t["net_pips"] > 0).mean(),
                      "avg win pips": t.loc[t["net_pips"] > 0, "net_pips"].mean(), "avg loss pips": t.loc[t["net_pips"] <= 0, "net_pips"].mean(),
                      "% -12 stop": reasons.get("stop", 0.0), "% trailing": reasons.get("trail", 0.0),
                      "% time / data end": reasons.get("time", 0.0) + reasons.get("end_of_data", 0.0),
                      "median minutes held": t["minutes"].median(), "max hours held": t["minutes"].max() / 60,
                      "worst trade pips": t["net_pips"].min(), "best trade pips": t["net_pips"].max()})
tab = trades.groupby(["structure", "variant"]).apply(per_structure)
display(tab.round(2)); tab.to_csv(OUT / "03d_per_structure.csv")

## 4. Net pips per trade, year by year

In [ ]:
yr = trades.pivot_table(index="year", columns=["structure", "variant"], values="net_pips", aggfunc="mean")
display(yr.round(2)); yr.to_csv(OUT / "03d_yearly_pips.csv")
fig, axes = plt.subplots(1, len(IDS), figsize=(4.4 * len(IDS), 3.6), sharey=True)
for ax, i in zip(axes, IDS):
    yr[i].plot.bar(ax=ax, width=0.8); ax.axhline(0, color="black", lw=0.5); ax.set_title(i); ax.set_xlabel("")
axes[0].set_ylabel("net pips per trade")
plt.tight_layout(); plt.savefig(OUT / "03d_yearly_pips.png", dpi=110); plt.show()

## 5. The compounding account: all four structures together, from $6,000

Equity is realised at each trade's exit (open trades are not marked, so long `TRAIL ONLY` holds understate drawdowns).

In [ ]:
rows, curves = [], {}
for v in VARIANTS:
    tv = trades[trades["variant"] == v]
    for L in LEVERAGES:
        acc = F.simulate_events(tv, L, E0)
        curves[(v, L)] = acc
        rows.append({"variant": v, "L each": L, **F.account_stats(acc, E0)})
res = pd.DataFrame(rows)
show = res.copy()
for c in ("cagr", "ann_vol", "max_drawdown", "worst_year", "best_year"):
    show[c] = (100 * show[c]).round(1).astype(str) + "%"
show["final_equity"] = show["final_equity"].round(0)
display(show); res.to_csv(OUT / "03d_account.csv", index=False)
fig, axes = plt.subplots(1, len(VARIANTS), figsize=(17, 4.2), sharey=True)
for ax, v in zip(axes, VARIANTS):
    for L in LEVERAGES:
        ax.plot(curves[(v, L)]["equity"].clip(lower=1), label=f"{L}x each")
    ax.axhline(E0, color="grey", lw=0.6); ax.axvline(pd.Timestamp("2026-01-01"), color="black", ls=":", lw=0.8)
    ax.set_yscale("log"); ax.set_title(v); ax.legend()
axes[0].set_ylabel("equity USD (log)")
plt.tight_layout(); plt.savefig(OUT / "03d_equity.png", dpi=110); plt.show()

## 6. Account return year by year (%)

In [ ]:
def yearly_returns(acc):
    y = acc["equity"].resample("YE").last()
    r = y.pct_change(); r.iloc[0] = y.iloc[0] / E0 - 1
    r.index = r.index.year
    return r
yrs = pd.DataFrame({(v, f"{L}x"): yearly_returns(curves[(v, L)]) for v in VARIANTS for L in LEVERAGES})
display((100 * yrs).round(1)); yrs.to_csv(OUT / "03d_yearly_returns.csv")

## 7. The same accounts without EURUSD-01 (the 03b set)

EURUSD-01 failed its 2021–2025 confirmation; this shows what including it costs.

In [ ]:
rows3 = []
for v in VARIANTS:
    tv = trades[(trades["variant"] == v) & (trades["structure"] != "EURUSD-01")]
    for L in LEVERAGES:
        st = F.account_stats(F.simulate_events(tv, L, E0), E0)
        rows3.append({"variant": v, "L each": L, "final_equity": round(st["final_equity"]), "cagr %": round(100 * st["cagr"], 1),
                      "max drawdown %": round(100 * st["max_drawdown"], 1), "worst year %": round(100 * st["worst_year"], 1)})
display(pd.DataFrame(rows3)); pd.DataFrame(rows3).to_csv(OUT / "03d_account_without_EURUSD01.csv", index=False)

In [ ]:
summary = {"study": "FXS03d", "descriptive": True, "synthetic": SYNTHETIC, "structures": IDS, "equity0": E0,
           "rule": {"initial": RULE.initial, "activate": RULE.activate, "distance": RULE.distance},
           "commission_usd_per_lot_rt": COMMISSION_PER_LOT, "spread_factor": SPREAD_FACTOR,
           "per_structure": tab.round(4).reset_index().to_dict(orient="records"), "account": res.to_dict(orient="records"),
           "note": "Stop settings chosen after seeing the data; hindsight structure set; 2026 already seen."}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", OUT / "summary.json")